# Sign Language MNIST — Data Cleaning, EDA & CNN Training

This notebook is the data-science companion to the **Sign Language → Text → Speech**
project. It covers:

1. **Loading** the Kaggle *Sign Language MNIST* dataset
2. **Data cleaning** & validation
3. **Exploratory data analysis** and **visualisations**
4. **Preparing** the data for modelling
5. **Training** the CNN used by the live app (saved to `models/sign_cnn.keras`)
6. **Evaluation** — accuracy, classification report, confusion matrix

> **Dataset:** [Sign Language MNIST](https://www.kaggle.com/datasets/datamunge/sign-language-mnist)
> — 28×28 grayscale images of ASL fingerspelling letters (A–Y, excluding the
> motion-based **J** and **Z**).

**Get the data first** (from the project root):

```bash
python -m src.download_data
# or: kaggle datasets download -d datamunge/sign-language-mnist -p data --unzip
```


## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# Make the project's ``src`` package importable from the notebooks/ folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.data_loader import load_raw, clean, to_arrays

print("Project root:", PROJECT_ROOT)
print("Image size:", config.IMG_SIZE, "| classes:", config.NUM_CLASSES)


## 2. Load the data

Each row holds a `label` (0–24) and 784 pixel columns (`pixel1`…`pixel784`)
that flatten a 28×28 grayscale image. Values range 0–255.


In [ ]:
train_raw = load_raw(config.TRAIN_CSV)
test_raw = load_raw(config.TEST_CSV)

print("Raw train shape:", train_raw.shape)
print("Raw test shape :", test_raw.shape)
train_raw.head()


In [ ]:
# Quick structural overview
print("Columns:", train_raw.shape[1], "(1 label + 784 pixels)")
print("\nLabel range:", train_raw['label'].min(), "->", train_raw['label'].max())
print("Pixel range:", int(train_raw.iloc[:, 1:].min().min()),
      "->", int(train_raw.iloc[:, 1:].max().max()))
train_raw.iloc[:, :6].describe()


## 3. Data cleaning

Sign Language MNIST is already well-formed, so cleaning is mostly **validation**:

- check for missing values
- check for duplicate rows
- confirm every label is in the valid range
- confirm every pixel is in 0–255

We then reuse the project's `clean()` helper so the notebook and the live app
apply *exactly* the same cleaning.


In [ ]:
# Missing values?
missing = train_raw.isna().sum().sum()
print("Total missing values:", missing)

# Duplicate rows?
dups = train_raw.duplicated().sum()
print("Duplicate rows:", dups)

# Labels outside the expected range?
bad_labels = train_raw.loc[
    (train_raw['label'] < 0) | (train_raw['label'] >= config.NUM_CLASSES)
]
print("Rows with out-of-range labels:", len(bad_labels))

# Pixels outside 0-255?
px = train_raw.iloc[:, 1:]
out_of_range = ((px < 0) | (px > 255)).any(axis=1).sum()
print("Rows with out-of-range pixels:", out_of_range)


In [ ]:
train = clean(train_raw)
test = clean(test_raw)

print("Clean train shape:", train.shape)
print("Clean test shape :", test.shape)
print("Rows removed (train):", len(train_raw) - len(train))


## 4. Exploratory data analysis

### 4.1 Class distribution

Which letters are well represented? J and Z are absent by design (they are
dynamic gestures). We map numeric labels to letters with `config.label_to_letter`.


In [ ]:
train = train.assign(letter=train['label'].map(config.label_to_letter))

counts = train['letter'].value_counts().sort_index()
plt.figure(figsize=(12, 4))
sns.barplot(x=counts.index, y=counts.values, color="#4C72B0")
plt.title("Samples per ASL letter (train)")
plt.xlabel("Letter"); plt.ylabel("Count")
plt.tight_layout(); plt.show()

print("Letters present:", list(counts.index))
print("Missing by design:", sorted(config.MISSING_LETTERS))
print(f"Min/Max per class: {counts.min()} / {counts.max()}  "
      f"(ratio {counts.max() / counts.min():.2f})")


### 4.2 What does a sign look like?

Reshape the flat pixel vectors back to 28×28 and view a grid of examples.


In [ ]:
def to_image(row):
    return row.drop(['label', 'letter']).to_numpy(dtype='uint8').reshape(
        config.IMG_SIZE, config.IMG_SIZE)

sample = train.groupby('label').head(1).sort_values('label')
fig, axes = plt.subplots(4, 6, figsize=(11, 8))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    ax.imshow(to_image(row), cmap='gray')
    ax.set_title(row['letter'], fontsize=11)
    ax.axis('off')
for ax in axes.ravel()[len(sample):]:
    ax.axis('off')
fig.suptitle("One example per class", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


### 4.3 The 'average' sign per letter

Averaging every image of a class reveals its prototype shape — and shows how
visually similar some letters are (a hint at which ones the model may confuse).


In [ ]:
mean_imgs = (train.drop(columns=['letter'])
             .groupby('label').mean())
labels_sorted = sorted(mean_imgs.index)

fig, axes = plt.subplots(4, 6, figsize=(11, 8))
for ax, label in zip(axes.ravel(), labels_sorted):
    img = mean_imgs.loc[label].to_numpy().reshape(config.IMG_SIZE, config.IMG_SIZE)
    ax.imshow(img, cmap='magma')
    ax.set_title(config.label_to_letter(label), fontsize=11)
    ax.axis('off')
for ax in axes.ravel()[len(labels_sorted):]:
    ax.axis('off')
fig.suptitle("Mean image per letter", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


### 4.4 Pixel-intensity distribution

How bright are the images overall? This guides normalisation — we scale pixels
to [0, 1] before training.


In [ ]:
flat = train.drop(columns=['label', 'letter']).to_numpy().ravel()
plt.figure(figsize=(10, 4))
plt.hist(flat, bins=50, color="#55A868")
plt.title("Pixel intensity distribution (all train images)")
plt.xlabel("Pixel value (0-255)"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()

print(f"Mean pixel: {flat.mean():.1f} | std: {flat.std():.1f}")


### 4.5 Average brightness per letter

A compact view of how letters differ in overall ink/lightness.


In [ ]:
brightness = (train.drop(columns=['letter'])
              .groupby('label').mean().mean(axis=1))
brightness.index = [config.label_to_letter(i) for i in brightness.index]
plt.figure(figsize=(12, 4))
sns.barplot(x=brightness.index, y=brightness.values, color="#C44E52")
plt.title("Mean brightness per letter")
plt.xlabel("Letter"); plt.ylabel("Average pixel value")
plt.tight_layout(); plt.show()


## 5. Prepare data for modelling

Reshape to `(n, 28, 28, 1)`, scale to `[0, 1]` (handled by `to_arrays`), and
keep the original train/test split provided by the dataset.


In [ ]:
X_train, y_train = to_arrays(train.drop(columns=['letter']))
X_test, y_test = to_arrays(test)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape, "| y_test :", y_test.shape)
print("Pixel range after scaling:", X_train.min(), "->", X_train.max())


## 6. Train the CNN

We reuse `src.train_model.build_model()` so the notebook trains the *exact same*
network the live app loads. TensorFlow/Keras is required here:

```bash
pip install tensorflow
```

> Training ~15 epochs on CPU takes a few minutes; with a GPU it's seconds.
> Reduce `EPOCHS` if you just want a quick run-through.


In [ ]:
from tensorflow import keras
from src.train_model import build_model

EPOCHS = 15
BATCH_SIZE = 128

# build_model() bakes augmentation in as Keras 3 layers (RandomRotation /
# Translation / Zoom) that are active only during training. No RandomFlip:
# a mirrored sign is a different gesture.
model = build_model()
model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True,
                                  monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
]

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    validation_data=(X_test, y_test),
    epochs=EPOCHS, callbacks=callbacks,
)


In [ ]:
# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(history.history['loss'], label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.set_xlabel('epoch'); ax2.legend()
plt.tight_layout(); plt.show()


## 7. Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {acc:.4f}  (loss {loss:.4f})")

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
present = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [config.label_to_letter(i) for i in present]
print(classification_report(y_test, y_pred, labels=present,
                            target_names=target_names))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=present)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, xticklabels=target_names, yticklabels=target_names,
            cmap='Blues', square=True, cbar_kws={'shrink': 0.7})
plt.title('Confusion matrix (test set)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout(); plt.show()


## 8. Save the model

The live app and the real-time demo load this file.


In [ ]:
config.MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(config.MODEL_PATH)
print("Saved ->", config.MODEL_PATH)


## 9. Conclusion

- The dataset is clean and well-balanced across the 24 static ASL letters.
- A compact CNN with light augmentation reaches high test accuracy.
- The confusion matrix highlights visually similar letters (e.g. **M/N**,
  **U/V/R**) — the hardest pairs to tell apart.
- The saved model powers both the **Streamlit dashboard** and the
  **real-time OpenCV demo**, where predicted letters are assembled into
  sentences and spoken with **Supertonic** TTS.

**Next steps:** collect a few self-captured webcam crops to fine-tune on, since
real cameras differ from the dataset's clean, centred images.
